# T35 - 不同曲线投资顺序分析
## 第4章：策略生成

### 概述
本章生成投资策略组合，从三个维度进行分析：
1. **品种维度**：不同债券类型的投资顺序（利率债/金融债/信用债）
2. **久期维度**：不同期限的投资顺序（短期/中长期/长期/超长期）
3. **资质维度**：不同评级的投资顺序（高资质/中资质/弱资质）

### 核心概念
- **策略（Strategy）**：单个债券投资决策
- **策略序列（StrategySequence）**：按时间顺序执行的一系列策略
- **维度（Dimension）**：策略变化的维度

In [ ]:
# 1. 导入必要的库
import pandas as pd
import numpy as np
from itertools import permutations
from datetime import datetime, timedelta
import os
import warnings

warnings.filterwarnings('ignore')

# 设置显示选项
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("库导入成功！")

In [ ]:
# 2. 加载配置
from config import (
    TYPE_MAPPING,
    RATING_MAPPING,
    DURATION_MAP,
    INITIAL_AMOUNT,
    OUTPUT_DIR,
    YIELD_DATA_FILE
)

print(f"初始投资金额: {INITIAL_AMOUNT:,.0f} 元")

In [ ]:
# 3. 定义策略类

class Strategy:
    """
    债券投资策略类
    
    Attributes:
    -----------
    bond_type : str
        债券类型（利率债/金融债/信用债）
    term : str
        期限分类（短期/中短期/中长期/长期/超长期）
    credit_level : str, optional
        资质等级（高资质/中资质/弱资质）
    """
    
    def __init__(self, bond_type, term, credit_level=None):
        """
        初始化策略
        
        Parameters:
        -----------
        bond_type : str
            债券类型
        term : str
            期限分类
        credit_level : str, optional
            资质等级
        """
        self.bond_type = bond_type
        self.term = term
        self.credit_level = credit_level
    
    def __str__(self):
        """字符串表示"""
        if self.credit_level:
            return f"{self.bond_type}_{self.term}_{self.credit_level}资质"
        return f"{self.bond_type}_{self.term}"
    
    def __repr__(self):
        return self.__str__()
    
    def to_dict(self):
        """转换为字典"""
        return {
            'bond_type': self.bond_type,
            'term': self.term,
            'credit_level': self.credit_level
        }
    
    def get_duration(self):
        """获取修正久期"""
        return DURATION_MAP.get(self.term, 4.0)

print("Strategy 类已定义")

In [ ]:
# 4. 定义策略生成器类

class StrategyGenerator:
    """
    策略生成器类
    
    用于生成三个维度的投资策略组合
    """
    
    def __init__(self):
        self.type_mapping = TYPE_MAPPING
        self.rating_mapping = RATING_MAPPING
    
    def generate_type_strategies(self):
        """
        生成品种维度策略
        
        固定：久期=中长期，资质=高资质
        变化：利率债/金融债/信用债的投资顺序
        
        Returns:
        --------
        list of list
            策略序列列表，每个策略序列包含3个策略
        """
        # 基础策略组合
        base_strategies = [
            ('利率债', '中长期', None),
            ('金融债', '中长期', '高资质'),
            ('信用债', '中长期', '高资质')
        ]
        
        # 生成所有排列
        strategies = []
        for perm in permutations(base_strategies):
            strategies.append(list(perm))
        
        return strategies
    
    def generate_term_strategies(self):
        """
        生成久期维度策略
        
        固定：品种=利率债或信用债，资质=中资质
        变化：久期顺序（长久期优先 vs 短久期优先）
        
        Returns:
        --------
        list of list
            策略序列列表
        """
        # 长久期优先策略
        long_first = [
            ('利率债', '超长期', None),
            ('信用债', '长期', '中资质'),
            ('信用债', '中长期', '中资质'),
            ('信用债', '短期', '中资质')
        ]
        
        # 短久期优先策略
        short_first = [
            ('信用债', '短期', '中资质'),
            ('信用债', '中长期', '中资质'),
            ('信用债', '长期', '中资质'),
            ('利率债', '超长期', None)
        ]
        
        return [long_first, short_first]
    
    def generate_credit_strategies(self):
        """
        生成资质维度策略
        
        固定：品种=信用债，久期=中长期
        变化：资质顺序（高资质优先 vs 弱资质优先）
        
        Returns:
        --------
        list of list
            策略序列列表
        """
        # 高资质优先
        high_first = [
            ('信用债', '中长期', '高资质'),
            ('信用债', '中长期', '中资质'),
            ('信用债', '中长期', '弱资质')
        ]
        
        # 弱资质优先
        low_first = [
            ('信用债', '中长期', '弱资质'),
            ('信用债', '中长期', '中资质'),
            ('信用债', '中长期', '高资质')
        ]
        
        return [high_first, low_first]
    
    def generate_all_strategies(self):
        """
        生成所有维度的策略组合
        
        Returns:
        --------
        dict
            按维度分类的策略组合字典
        """
        return {
            '品种维度': self.generate_type_strategies(),
            '久期维度': self.generate_term_strategies(),
            '资质维度': self.generate_credit_strategies()
        }

# 创建策略生成器
generator = StrategyGenerator()
print("StrategyGenerator 类已定义")

In [ ]:
# 5. 生成并展示品种维度策略

def display_type_strategies():
    """展示品种维度策略"""
    strategies = generator.generate_type_strategies()
    
    print("品种维度策略")
    print("=" * 60)
    print(f"固定条件: 久期=中长期, 资质=高资质")
    print(f"策略数量: {len(strategies)}")
    print()
    
    for i, seq in enumerate(strategies, 1):
        strategy_names = [str(Strategy(*s)) for s in seq]
        print(f"策略 {i}: {' -> '.join(strategy_names)}")
    
    return strategies

type_strategies = display_type_strategies()

In [ ]:
# 6. 生成并展示久期维度策略

def display_term_strategies():
    """展示久期维度策略"""
    strategies = generator.generate_term_strategies()
    
    print("久期维度策略")
    print("=" * 60)
    print(f"固定条件: 品种=利率债/信用债, 资质=中资质")
    print(f"策略数量: {len(strategies)}")
    print()
    
    strategy_names = ['长久期优先', '短久期优先']
    for name, seq in zip(strategy_names, strategies):
        strategy_str = [str(Strategy(*s)) for s in seq]
        print(f"{name}:")
        print(f"  {' -> '.join(strategy_str)}")
        print()
    
    return strategies

term_strategies = display_term_strategies()

In [ ]:
# 7. 生成并展示资质维度策略

def display_credit_strategies():
    """展示资质维度策略"""
    strategies = generator.generate_credit_strategies()
    
    print("资质维度策略")
    print("=" * 60)
    print(f"固定条件: 品种=信用债, 久期=中长期")
    print(f"策略数量: {len(strategies)}")
    print()
    
    strategy_names = ['高资质优先', '弱资质优先']
    for name, seq in zip(strategy_names, strategies):
        strategy_str = [str(Strategy(*s)) for s in seq]
        print(f"{name}:")
        print(f"  {' -> '.join(strategy_str)}")
        print()
    
    return strategies

credit_strategies = display_credit_strategies()

In [ ]:
# 8. 策略汇总

def generate_strategy_summary():
    """生成策略汇总"""
    all_strategies = generator.generate_all_strategies()
    
    print("策略汇总")
    print("=" * 60)
    
    summary_data = []
    for dimension, strategy_list in all_strategies.items():
        print(f"\n{dimension}:")
        print(f"  策略数量: {len(strategy_list)}")
        print(f"  每个策略的步骤数: {len(strategy_list[0])}")
        
        summary_data.append({
            '维度': dimension,
            '策略数量': len(strategy_list),
            '步骤数': len(strategy_list[0]),
            '总组合数': len(strategy_list) * len(strategy_list[0])
        })
    
    summary_df = pd.DataFrame(summary_data)
    print("\n策略统计汇总:")
    print(summary_df.to_string(index=False))
    
    print(f"\n总计策略序列数: {summary_df['策略数量'].sum()}")
    
    return all_strategies, summary_df

all_strategies, strategy_summary = generate_strategy_summary()

In [ ]:
# 9. 定义周期分割函数

def split_period(start_date, end_date, n_splits):
    """
    将时间段等分为n个子期间
    
    Parameters:
    -----------
    start_date : datetime or str
        起始日期
    end_date : datetime or str
        结束日期
    n_splits : int
        分割份数
        
    Returns:
    --------
    list of tuples
        每个元组包含子期间的(起始日期, 结束日期)
    """
    # 确保输入的是datetime类型
    if isinstance(start_date, str):
        start_date = pd.to_datetime(start_date)
    if isinstance(end_date, str):
        end_date = pd.to_datetime(end_date)
    
    # 计算总天数
    total_days = (end_date - start_date).days
    # 计算每个子期间的天数
    days_per_split = total_days // n_splits
    
    # 生成子期间
    periods = []
    for i in range(n_splits):
        if i == n_splits - 1:
            # 最后一个子期间直接使用结束日期，避免舍入误差
            period_end = end_date
        else:
            period_end = start_date + pd.Timedelta(days=(i + 1) * days_per_split)
        
        period_start = start_date + pd.Timedelta(days=i * days_per_split)
        periods.append((period_start, period_end))
    
    return periods

# 测试周期分割函数
test_start = '2024-01-01'
test_end = '2024-12-31'
test_periods = split_period(test_start, test_end, 4)

print("测试周期分割函数")
print("=" * 60)
print(f"总周期: {test_start} ~ {test_end}")
print(f"分割数: 4")
print(f"\n子期间:")
for i, (start, end) in enumerate(test_periods, 1):
    days = (end - start).days
    print(f"  {i}. {start.strftime('%Y-%m-%d')} ~ {end.strftime('%Y-%m-%d')} ({days}天)")

In [ ]:
# 10. 保存策略配置

def save_strategy_config(all_strategies):
    """保存策略配置"""
    # 转换为可序列化格式
    config_data = {
        'version': '1.0',
        'created_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'dimensions': {}
    }
    
    for dimension, strategy_list in all_strategies.items():
        config_data['dimensions'][dimension] = {
            'strategy_count': len(strategy_list),
            'strategies': []
        }
        
        for i, seq in enumerate(strategy_list, 1):
            strategy_seq = {
                'id': i,
                'name': ' -> '.join([str(Strategy(*s)) for s in seq]),
                'steps': [{'bond_type': s[0], 'term': s[1], 'credit_level': s[2]} for s in seq]
            }
            config_data['dimensions'][dimension]['strategies'].append(strategy_seq)
    
    # 保存JSON文件
    import json
    config_file = os.path.join(OUTPUT_DIR, '策略配置.json')
    with open(config_file, 'w', encoding='utf-8') as f:
        json.dump(config_data, f, ensure_ascii=False, indent=2)
    
    print(f"策略配置已保存: {config_file}")
    
    # 保存策略汇总CSV
    summary_file = os.path.join(OUTPUT_DIR, '策略汇总.csv')
    strategy_summary.to_csv(summary_file, index=False, encoding='utf-8-sig')
    print(f"策略汇总已保存: {summary_file}")
    
    return config_data

config_data = save_strategy_config(all_strategies)

### 总结

本章完成了以下工作：
1. 定义了 `Strategy` 类表示单个债券投资策略
2. 定义了 `StrategyGenerator` 类生成三个维度的策略组合
3. 生成了品种维度策略（6种排列）
4. 生成了久期维度策略（2种）
5. 生成了资质维度策略（2种）
6. 定义了周期分割函数
7. 保存了策略配置文件

**策略统计:**
- 品种维度: 6种策略序列
- 久期维度: 2种策略序列
- 资质维度: 2种策略序列
- 总计: 10种策略序列

**输出文件:**
- `策略配置.json`: 策略配置信息
- `策略汇总.csv`: 策略统计汇总

**下一步**: 运行 `05-回测分析.ipynb` 进行策略回测和收益计算。